# VGG16 CIFAR10 Training: TF32 Base + FP8 Low Precision

Notebook ini melatih model **VGG16** pada dataset **CIFAR-10** menggunakan konfigurasi mixed-precision:

| Komponen | Precision | Keterangan |
|----------|-----------|------------|
| **Base precision** | TF32 | Diaktifkan via `torch.backends.cuda.matmul.allow_tf32 = True`. TF32 menggunakan 10-bit mantissa (vs 23-bit FP32) dengan exponent range FP32 penuh. |
| **Low precision** | FP8 (E4M3) | Diterapkan pada Conv2d dan Linear layers terpilih menggunakan native FP8 casting + delayed scaling. |

### Arsitektur Precision Assignment
- **Conv layers 0-6** (64→64→128→128→256→256→256): FP8 — layer-layer awal yang compute-intensive
- **Conv layers 7-12** (512 channels): TF32 — layer akhir yang lebih sensitif terhadap precision
- **Linear layer 0** (512→512): FP8 — FC layer pertama yang paling besar
- **Linear layers 1-2**: TF32 — classifier layers yang perlu precision lebih tinggi

### Key Concepts
- **Delayed Scaling**: Scale factor dihitung dari amax history (16 iterasi), bukan per-batch
- **FP8 E4M3**: 4-bit exponent, 3-bit mantissa, range ±448 — cocok untuk forward activations & weights
- **NativePrecisionMixin**: Setiap layer bisa di-assign precision mode secara independen

In [ ]:
# ============================================================
# Cell 2: Imports dan Environment Check
# ============================================================
import sys
import os
import time
import warnings

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Tambahkan project root ke sys.path agar ext3 bisa diimport
PROJECT_ROOT = os.path.abspath(os.path.dirname('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import ext3 native precision layers
from ext3.nn.nn_native import (
    NativeConv2d,
    NativeLinear,
    NativeBatchNorm2d,
    NativeReLU,
    NativeMaxPool2d,
    NativeAdaptiveAvgPool2d,
    NativeDropout,
)

# Import precision management utilities
from ext3.core.include.native_precision import (
    NativePrecisionMode,
    check_fp8_support,
    enable_tf32,
    FP8Config,
)
from ext3.core.include.pasn import NativePasn


# ---- Environment Check ----
print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"PyTorch version  : {torch.__version__}")
print(f"Torchvision ver  : {torchvision.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name         : {torch.cuda.get_device_name(0)}")
    cap = torch.cuda.get_device_capability(0)
    print(f"Compute cap      : {cap[0]}.{cap[1]}")
    print(f'GPU memory       : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    print(f"FP8 supported    : {check_fp8_support()}")
else:
    print("WARNING: CUDA not available! Training will be very slow on CPU.")

print(f"Python version   : {sys.version.split()[0]}")


In [ ]:
# ============================================================
# Cell 3: Global Configuration
# ============================================================

# Aktifkan TF32 untuk semua matmul dan convolution operations
enable_tf32()

CONFIG = {
    'batch_size': 128,
    'epochs': 50,
    'lr': 0.01,
    'momentum': 0.9,
    'weight_decay': 5e-4,
    'num_workers': 0,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # --- Native APA (Adaptive Precision Assignment) Config ---
    # Diambil dari script_ext3/cifar_perf_main_2_heuris.sh
    'pa_upd_schm': 'topr_dec',    # Demote layer terbesar lebih dulu
    'pa_upd_rmin': 0.3,           # Target 30% dari total parameter turun ke FP8
    'pa_upd_rmax': 0.4,           
    'spike_threshold': 0.5,       # Ambang batas lonjakan AMAX (50% dari rata-rata). 
                                  # Ekuivalen dengan --pa_ovr_thrs 0.01 pada simulasi bit asli.
}

print("\n=== Training Configuration ===")
for k, v in CONFIG.items():
    print(f"  {k:20s}: {v}")

print(f"\n  TF32 matmul        : {torch.backends.cuda.matmul.allow_tf32}")
print(f"  TF32 cudnn         : {torch.backends.cudnn.allow_tf32}")


In [ ]:
# ============================================================
# Cell 4: VGG16 Model Definition
# ============================================================
# Menggunakan ext3 Native layers agar setiap Conv2d/Linear bisa
# di-assign precision mode secara independen (TF32 atau FP8).
# Diadaptasi untuk CIFAR-10 (input 32x32, output 10 classes).

# VGG16 feature config: angka = jumlah filter, 'M' = MaxPool2d
VGG16_CFG = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M']


class VGG16Native(nn.Module):
    """
    VGG16 dengan NativeConv2d/NativeLinear untuk mixed-precision.
    
    Struktur:
      features: 13 NativeConv2d + NativeBatchNorm2d + NativeReLU + 5 NativeMaxPool2d
      avgpool : NativeAdaptiveAvgPool2d(1, 1) — output 512x1x1
      classifier: 3 NativeLinear (512→512→512→10)
    """

    def __init__(self, num_classes: int = 10):
        super(VGG16Native, self).__init__()
        
        # Build feature extractor dari VGG16 config
        self.features = self._make_layers(VGG16_CFG)
        
        # AdaptiveAvgPool menangani arbitrary input size → 1x1
        # Untuk CIFAR-10 (32x32), setelah 5 MaxPool: 32→16→8→4→2→1
        self.avgpool = NativeAdaptiveAvgPool2d((1, 1))
        
        # Classifier: 3 fully-connected layers
        # Original VGG: 512*7*7→4096→4096→1000
        # CIFAR-10 adapted: 512→512→512→10
        self.classifier = nn.Sequential(
            NativeLinear(512, 512),
            NativeReLU(inplace=False),
            NativeDropout(p=0.5),
            NativeLinear(512, 512),
            NativeReLU(inplace=False),
            NativeDropout(p=0.5),
            NativeLinear(512, num_classes),
        )
        
        # Weight initialization (Kaiming untuk Conv, Xavier untuk Linear)
        self._initialize_weights()

    def _make_layers(self, cfg):
        """Build feature extractor dari VGG config list."""
        layers = []
        in_channels = 3  # CIFAR-10: RGB input
        
        for v in cfg:
            if v == 'M':
                layers.append(NativeMaxPool2d(kernel_size=2, stride=2))
            else:
                # Conv2d → BatchNorm2d → ReLU
                layers.append(NativeConv2d(in_channels, v, kernel_size=3, padding=1))
                layers.append(NativeBatchNorm2d(v))
                layers.append(NativeReLU(inplace=False))
                in_channels = v
        
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        """Inisialisasi weights menggunakan Kaiming/Xavier."""
        for m in self.modules():
            if isinstance(m, NativeConv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, NativeBatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, NativeLinear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)  # Flatten: [B, 512, 1, 1] → [B, 512]
        x = self.classifier(x)
        return x


# Quick architecture verification
print("=== VGG16Native Architecture ===")
_model_test = VGG16Native(num_classes=10)

# Count layers
_n_conv = sum(1 for m in _model_test.modules() if isinstance(m, NativeConv2d))
_n_linear = sum(1 for m in _model_test.modules() if isinstance(m, NativeLinear))
_n_params = sum(p.numel() for p in _model_test.parameters())

print(f"  NativeConv2d layers : {_n_conv}")
print(f"  NativeLinear layers : {_n_linear}")
print(f"  Total parameters    : {_n_params:,}")
print(f"  Parameters (MB)     : {_n_params * 4 / 1024**2:.1f}")
del _model_test

In [ ]:
# ============================================================
# Cell 5: Precision Assignment via NativePasn
# ============================================================

def assign_precision(model: nn.Module, config: dict) -> NativePasn:
    """
    Menggunakan NativePasn untuk melakukan Tensor Grouping dan Precision Demotion.
    Melindungi layer pertama dan terakhir secara otomatis.
    """
    print("\n" + "=" * 60)
    print("PRECISION ASSIGNMENT (Native APA)")
    print("=" * 60)
    
    # 1. Initialize Pasn (Fase Grouping)
    pasn = NativePasn(model)
    
    # 2. Set global spike threshold untuk amax tracking
    from ext3.nn.nn_native import get_fp8_manager
    manager = get_fp8_manager()
    manager.set_spike_threshold(config['spike_threshold'])
    
    # 3. Apply Demotion
    stats = pasn.apply_demotion(
        scheme=config['pa_upd_schm'],
        r_min=config['pa_upd_rmin'],
        r_max=config['pa_upd_rmax'],
        base_mode=NativePrecisionMode.BASE_TF32,
        low_mode=NativePrecisionMode.LOW_FP8,
        protect_ends=True  # Sesuai dengan pa_upd_no_end
    )
    
    print(f"  Scheme          : {stats.get('scheme')}")
    print(f"  Target Ratio    : {stats.get('target_ratio')}")
    print(f"  Actual Ratio    : {stats.get('actual_ratio'):.3f}")
    print(f"  Demoted Layers  : {stats.get('demoted_layers')} / {stats.get('total_layers')}")
    print("=" * 60)
    
    return pasn

print("assign_precision() defined ✓")


In [ ]:
# ============================================================
# Cell 6: CIFAR-10 Data Pipeline
# ============================================================
# Standard data augmentation untuk CIFAR-10:
# - RandomCrop(32, padding=4): spatial augmentation
# - RandomHorizontalFlip: mirror augmentation
# - Normalize: channel-wise mean/std normalization

# CIFAR-10 channel statistics (precomputed)
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

# Training transforms: augmentation + normalization
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Test transforms: normalization only
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Download & create datasets
trainset = torchvision.datasets.CIFAR10(
    root="./dataset/cifar10", train=True, download=True, transform=transform_train
)
testset = torchvision.datasets.CIFAR10(
    root="./dataset/cifar10", train=False, download=True, transform=transform_test
)

# Create dataloaders
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
)
testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
)

# CIFAR-10 class names
CLASSES = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"\n=== CIFAR-10 Dataset ===")
print(f"  Training samples   : {len(trainset):,}")
print(f"  Test samples       : {len(testset):,}")
print(f"  Batch size         : {CONFIG['batch_size']}")
print(f"  Training batches   : {len(trainloader)}")
print(f"  Test batches       : {len(testloader)}")
print(f"  Classes            : {len(CLASSES)}")

In [ ]:
# ============================================================
# Cell 7: VRAM Monitoring Utility
# ============================================================

def get_vram_mb() -> float:
    """Return current GPU memory allocated in MB."""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0.0


def get_vram_peak_mb() -> float:
    """Return peak GPU memory allocated in MB."""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return 0.0


def print_vram_status(label: str = ""):
    """Print current and peak VRAM usage."""
    current = get_vram_mb()
    peak = get_vram_peak_mb()
    prefix = f"[{label}] " if label else ""
    print(f"  {prefix}VRAM: {current:.1f} MB (current) / {peak:.1f} MB (peak)")


print("VRAM monitoring utilities defined ✓")
if torch.cuda.is_available():
    print_vram_status("Initial")

In [ ]:
# ============================================================
# Cell 8: Training Function
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device, pasn_manager: NativePasn):
    """
    Train model untuk satu epoch.
    Termasuk pemanggilan Fase 3 (Precision Promotion) setelah setiap backward pass.
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    total_promotions = 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        # --- NATIVE APA: Precision Promotion ---
        # Mengecek apakah ada layer FP8 yang mengalami lonjakan amax ekstrim
        promoted = pasn_manager.check_and_promote(NativePrecisionMode.BASE_TF32)
        total_promotions += promoted
        
        # Accumulate statistics
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy, total_promotions

print("train_epoch() defined ✓")


In [ ]:
# ============================================================
# Cell 9: Evaluation Function
# ============================================================

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """
    Evaluate model pada test/validation set.
    
    Menggunakan torch.no_grad() untuk menghemat memory dan
    mempercepat inference (tidak perlu menyimpan gradients).
    
    Args:
        model: VGG16Native model
        loader: Test DataLoader
        criterion: Loss function
        device: 'cuda' atau 'cpu'
    
    Returns:
        tuple: (avg_loss, accuracy_percent)
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy


print("evaluate() defined ✓")

In [ ]:
# ============================================================
# Cell 10: Main Training Loop
# ============================================================

# ---- 1. Initialize Model ----
print("Initializing VGG16Native model...")
model = VGG16Native(num_classes=10)
model = model.to(CONFIG['device'])

if torch.cuda.is_available():
    print_vram_status("After model to GPU")

# ---- 2. Assign Precision ----
pasn_manager = assign_precision(model, CONFIG)

# ---- 3. Loss, Optimizer, Scheduler ----
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    lr=CONFIG['lr'],
    momentum=CONFIG['momentum'],
    weight_decay=CONFIG['weight_decay'],
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['epochs']
)

# ---- 4. Training History ----
history = {
    'train_loss': [],
    'train_acc': [],
    'test_loss': [],
    'test_acc': [],
    'vram_mb': [],
    'epoch_time': [],
    'lr': [],
    'promotions': [],
}

best_test_acc = 0.0

# ---- 5. Training Loop ----
print("\n" + "=" * 85)
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Test Acc':>8} | "
      f"{'Promoted':>8} | {'VRAM MB':>8} | {'Time':>6} | {'LR':>8}")
print("-" * 85)

total_train_start = time.time()

for epoch in range(1, CONFIG['epochs'] + 1):
    epoch_start = time.time()
    
    # Train
    train_loss, train_acc, promotions = train_epoch(
        model, trainloader, criterion, optimizer, CONFIG['device'], pasn_manager
    )
    
    # Evaluate
    test_loss, test_acc = evaluate(
        model, testloader, criterion, CONFIG['device']
    )
    
    # Step scheduler
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    # Record VRAM
    vram = get_vram_mb()
    epoch_time = time.time() - epoch_start
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    history['vram_mb'].append(vram)
    history['epoch_time'].append(epoch_time)
    history['lr'].append(current_lr)
    history['promotions'].append(promotions)
    
    # Track best
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_marker = " ★"
    else:
        best_marker = ""
    
    # Print epoch summary
    print(f"{epoch:5d} | {train_loss:10.4f} | {train_acc:8.2f}% | {test_acc:7.2f}% | "
          f"{vram:7.1f}M | {epoch_time:5.1f}s | {current_lr:.6f}{best_marker}")

total_time = time.time() - total_train_start

print("-" * 85)
print(f"\nTraining complete!")
print(f"  Total time       : {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  Best test acc    : {best_test_acc:.2f}%")
print(f"  Final train acc  : {history['train_acc'][-1]:.2f}%")
print(f"  Final test acc   : {history['test_acc'][-1]:.2f}%")
if torch.cuda.is_available():


In [ ]:
# ============================================================
# Cell 11: Visualization
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

# ---- Subplot 1: Loss Curve ----
ax1 = axes[0]
ax1.plot(epochs_range, history['train_loss'], 'b-', linewidth=1.5, label='Train Loss', alpha=0.8)
ax1.plot(epochs_range, history['test_loss'], 'r--', linewidth=1.5, label='Test Loss', alpha=0.8)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training & Test Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(1, len(history['train_loss']))

# ---- Subplot 2: Accuracy Curve ----
ax2 = axes[1]
ax2.plot(epochs_range, history['train_acc'], 'b-', linewidth=1.5, label='Train Acc', alpha=0.8)
ax2.plot(epochs_range, history['test_acc'], 'r--', linewidth=1.5, label='Test Acc', alpha=0.8)
ax2.axhline(y=best_test_acc, color='green', linestyle=':', alpha=0.5, label=f'Best: {best_test_acc:.1f}%')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training & Test Accuracy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10, loc='lower right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(1, len(history['train_acc']))

# ---- Subplot 3: VRAM Usage ----
ax3 = axes[2]
ax3.fill_between(epochs_range, history['vram_mb'], alpha=0.3, color='purple')
ax3.plot(epochs_range, history['vram_mb'], 'purple', linewidth=1.5, label='VRAM Allocated')
ax3.set_xlabel('Epoch', fontsize=12)
ax3.set_ylabel('VRAM (MB)', fontsize=12)
ax3.set_title('GPU VRAM Usage', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(1, len(history['vram_mb']))

# Global styling
fig.suptitle('VGG16 CIFAR-10: TF32 Base + FP8 Low Precision',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('vgg16_tf32_fp8_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to: vgg16_tf32_fp8_results.png")

## Analisis: TF32 Base + FP8 Low Precision

### Mengapa Kombinasi TF32 + FP8?

**TF32 sebagai Base Precision:**
- TF32 (TensorFloat-32) menggunakan 10-bit mantissa dengan exponent range FP32 penuh
- Pada GPU Ampere ke atas, TF32 memberikan **~2-3x speedup** untuk matmul dan convolution
- Tidak memerlukan perubahan kode apapun — cukup set global flag
- Akurasi sangat mendekati FP32 penuh untuk hampir semua use case

**FP8 sebagai Low Precision (Selected Layers):**
- FP8 E4M3 (4-bit exponent, 3-bit mantissa) memiliki range ±448
- Cocok untuk layer-layer awal (conv 0-6) yang memproses fitur low-level
- **Delayed scaling** menjaga stabilitas: scale factor dihitung dari 16-iterasi history
- Hanya applied pada forward pass; backward tetap FP32 untuk stabilitas gradient

### Strategi Layer Assignment

| Layer Group | Precision | Alasan |
|-------------|-----------|--------|
| Conv2d 0-6 (64-256 ch) | FP8 | Layer awal: toleran terhadap noise, high compute ratio |
| Conv2d 7-12 (512 ch) | TF32 | Layer akhir: semantic features sensitif terhadap precision |
| Linear 0 (512→512) | FP8 | FC layer terbesar, dominan compute |
| Linear 1-2 (512→10) | TF32 | Classifier output, perlu precision tinggi |

### Keuntungan Mixed Precision

1. **Memory Efficiency**: FP8 weights dan activations menggunakan 1 byte vs 4 bytes (FP32)
2. **Compute Throughput**: FP8 tensor cores memberikan throughput lebih tinggi
3. **Training Stability**: Delayed scaling mencegah overflow/underflow
4. **Selective Application**: Hanya layer yang toleran yang menggunakan FP8

### Catatan Penting
- FP8 memerlukan GPU dengan compute capability ≥ 8.9 (Ada Lovelace) atau ≥ 9.0 (Hopper)
- Pada GPU yang tidak mendukung FP8, framework otomatis fallback ke FP16
- BatchNorm selalu beroperasi di FP32 (best practice untuk running statistics)